In [1]:
# ============================================================
# BoneBot - NHANES 2013-2014 training pipeline (v2, evidence-refined)
#   Lightweight : P(osteoporosis) triage   (logistic: age + BMI + menopause)
#   Heavyweight : estimated T-score         (ridge)
#   Benchmark   : does menopause + wearable activity beat the age+weight baseline?
#
# Evidence (MOTIVATION.md, sec 2): for women, prediction "collapses to age + BMI" --
# exactly what OST / the existing tools use. Our differentiators are the two modalities
# the field DROPS: menopause status + objective wearable activity. So the winning claim
# is the ABLATION (do those add value over age+BMI?), not "we beat OST outright".
# ============================================================

import ssl, urllib.request, io
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge, LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import (mean_absolute_error, r2_score, roc_auc_score,
                             average_precision_score, roc_curve)

ssl._create_default_https_context = ssl._create_unverified_context

# --- 1. download + merge (all one row per person -> safe to merge on SEQN) ---
BASE = "https://wwwn.cdc.gov/Nchs/Data/Nhanes/Public/2013/DataFiles/"
HEADERS = {"User-Agent": "Mozilla/5.0"}

def load(f):
    req = urllib.request.Request(BASE + f + ".xpt", headers=HEADERS)
    with urllib.request.urlopen(req) as r:
        return pd.read_sas(io.BytesIO(r.read()), format="xport")

files = ["DEMO_H", "BMX_H", "DXXFEM_H", "RHQ_H", "OSQ_H", "SMQ_H",
         "ALQ_H", "MCQ_H", "VID_H", "BIOPRO_H", "PAQ_H"]
frames = {}
for f in files:
    try:
        frames[f] = load(f)
    except Exception as e:
        print("MISSING / renamed:", f, e)

merged = frames["DEMO_H"]
for f in files[1:]:
    if f in frames:
        merged = merged.merge(frames[f], on="SEQN", how="left")

REF_MEAN, REF_SD = 0.858, 0.120   # NHANES III young-adult white-female femoral-neck reference

# --- 2. heavyweight sample: women 50+ with a femur DXA scan ---
women = merged[(merged.RIAGENDR == 2) & (merged.RIDAGEYR >= 50) & (merged.DXXNKBMD.notna())].copy()

women["Tscore"]                 = (women["DXXNKBMD"] - REF_MEAN) / REF_SD      # target
women["age"]                    = women["RIDAGEYR"]
women["bmi"]                    = women["BMXBMI"]
women["vitaminD"]               = women["LBXVIDMS"]                            # nmol/L (evidence-backed nutritional)
women["calcium"]                = women["LBDSCASI"]                            # mmol/L (evidence-backed nutritional)
women["currentSmoker"]          = women["SMQ040"].isin([1, 2]).astype(int)
women["priorFragilityFracture"] = women[["OSQ010A","OSQ010B","OSQ010C"]].eq(1).any(axis=1).astype(int)
women["glucocorticoids"]        = (women["OSQ130"] == 1).astype(int)
women["highAlcohol"]            = (women["ALQ130"] >= 3).astype(int)
women["yearsSinceMenopause"]    = (women["RIDAGEYR"] - women["RHQ060"]).clip(lower=0)
# (dropped rbc + alp: no supporting evidence in MOTIVATION.md -> they were noise)

# wearable activity: mean daily MIMS from the wrist accelerometer, normalised 0-1
pax = load("PAXDAY_H")
act = pax.groupby("SEQN")["PAXMTSD"].mean().rename("daily_mims").reset_index()  # verify PAXMTSD
women = women.merge(act, on="SEQN", how="left")                                 # aggregated first -> no dup rows
women["activity_level"] = (women["daily_mims"] / women["daily_mims"].quantile(0.95)).clip(0, 1)

print("heavyweight sample:", women.shape[0], "women")

# --- 3. lightweight triage: P(osteoporosis) from age + BMI + menopause (BROAD age range) ---
# BMI added: the evidence says weight/BMI is the #1 co-predictor with age (that's what OST
# uses). Without it the triage cannot match OST.
fem = merged[(merged.RIAGENDR == 2) & (merged.DXXNKBMD.notna()) & (merged.RIDAGEYR >= 18)].copy()
fem["osteoporosis"]   = ((fem["DXXNKBMD"] - REF_MEAN) / REF_SD <= -2.5).astype(int)
fem["age"]            = fem["RIDAGEYR"]
fem["bmi"]            = fem["BMXBMI"]
fem["postmenopausal"] = ((fem["RIDAGEYR"] - fem["RHQ060"]) >= 0).astype(int)

triage_feats = ["age", "bmi", "postmenopausal"]
Xl = fem[triage_feats].apply(lambda c: c.fillna(c.median()))
yl = fem["osteoporosis"]
Xtr_l, Xte_l, ytr_l, yte_l = train_test_split(Xl, yl, test_size=0.25, random_state=0, stratify=yl)
triage = LogisticRegression(max_iter=1000).fit(Xtr_l, ytr_l)
probs_l = triage.predict_proba(Xte_l)[:, 1]
triage_auc = roc_auc_score(yte_l, probs_l)

def prob_osteoporosis(age, bmi, postmenopausal):
    row = pd.DataFrame([[age, bmi, postmenopausal]], columns=triage_feats)
    return triage.predict_proba(row)[0, 1]        # column 1 = P(osteoporosis)

# triage cutoff: keep high sensitivity like the existing tools (~89-94%).
# Refer everyone at/above the cutoff; safely exclude the confident-negatives below it.
fpr, tpr, thr = roc_curve(yte_l, probs_l)
target_sens = 0.95
i = np.where(tpr >= target_sens)[0][0]
cutoff = float(thr[i])
excluded = float((probs_l < cutoff).mean())

# --- 4. heavyweight: estimated T-score (ridge) ---
# Evidence-backed set: age + BMI (core, like the tools) + our differentiators
# (menopause, wearable activity) + FRAX factors + nutritional (vit D, calcium).
features = ["age", "bmi", "yearsSinceMenopause", "activity_level",
            "priorFragilityFracture", "glucocorticoids", "currentSmoker",
            "highAlcohol", "vitaminD", "calcium"]

X = women[features].apply(lambda c: c.fillna(c.median()))    # median-impute missing
y = women["Tscore"]
Xtr_h, Xte_h, ytr_h, yte_h = train_test_split(X, y, test_size=0.25, random_state=0)
model = Ridge(alpha=1.0).fit(Xtr_h, ytr_h)
pred_h = model.predict(Xte_h)

perf_table = pd.DataFrame({
    "Metric": ["MAE", "R2", "n_train", "n_test"],
    "Value":  [round(mean_absolute_error(yte_h, pred_h), 3),
               round(r2_score(yte_h, pred_h), 3), len(Xtr_h), len(Xte_h)],
})
coef_table = pd.DataFrame({"Term": ["intercept"] + features,
                           "Coefficient": [model.intercept_] + list(model.coef_)})
coef_table["Coefficient"] = coef_table["Coefficient"].round(4)
coef_table["Direction"]   = coef_table["Coefficient"].apply(
    lambda c: "raises T-score" if c > 0 else "lowers T-score")
coef_table = coef_table.sort_values("Coefficient").reset_index(drop=True)

# --- 5. benchmark ABLATION: does menopause + wearable activity beat age+weight? ---
# The honest scientific claim (MOTIVATION sec 3a/4d): the field drops menopause + activity;
# we show they ADD value over the age+BMI baseline the existing tools already use.
women["osteoporosis"] = (women["Tscore"] <= -2.5).astype(int)
yb_tr = women.loc[Xtr_h.index, "osteoporosis"]
yb_te = women.loc[Xte_h.index, "osteoporosis"]

ablation = {
    "baseline (age + BMI) ~ current tools": ["age", "bmi"],
    "+ menopause":                          ["age", "bmi", "yearsSinceMenopause"],
    "+ wearable activity":                  ["age", "bmi", "yearsSinceMenopause", "activity_level"],
    "full model":                           features,
}
print("\n=== Ablation: does our added signal beat the age+weight baseline? (osteoporosis classifier, same split) ===")
for name, feats in ablation.items():
    Xb = women[feats].apply(lambda c: c.fillna(c.median()))
    clf = LogisticRegression(max_iter=1000).fit(Xb.loc[Xtr_h.index], yb_tr)
    prob = clf.predict_proba(Xb.loc[Xte_h.index])[:, 1]
    print(f"   {name:38s}  AUC {roc_auc_score(yb_te, prob):.3f}   PR-AUC {average_precision_score(yb_te, prob):.3f}")

ost = women.loc[Xte_h.index].copy()
ost["ost"] = 0.2 * (ost["BMXWT"] - ost["age"])
mm = ost["ost"].notna()
print(f"   {'OST tool (age + weight, reference)':38s}  AUC {roc_auc_score(ost.loc[mm,'osteoporosis'], -ost.loc[mm,'ost']):.3f}")

# --- 6. results ---
print(f"\nLIGHTWEIGHT triage (age + BMI + menopause) - AUC: {triage_auc:.3f}")
print(f"Cutoff for {target_sens:.0%} sensitivity: refer if P(osteoporosis) >= {cutoff:.3f}  "
      f"(safely excludes {excluded:.0%} of women as low-risk)")
for a, b, mmeno in [(23, 25, 0), (43, 25, 0), (55, 24, 1), (70, 22, 1)]:
    print(f"   age {a}, bmi {b}, postmeno {mmeno}: P(osteoporosis) = {prob_osteoporosis(a, b, mmeno):.1%}")

print("\nHEAVYWEIGHT T-score model")
print(perf_table.to_string(index=False))
print()
print(coef_table.to_string(index=False))

heavyweight sample: 1119 women

=== Ablation: does our added signal beat the age+weight baseline? (osteoporosis classifier, same split) ===
   baseline (age + BMI) ~ current tools    AUC 0.775   PR-AUC 0.281
   + menopause                             AUC 0.776   PR-AUC 0.283
   + wearable activity                     AUC 0.781   PR-AUC 0.313
   full model                              AUC 0.782   PR-AUC 0.327
   OST tool (age + weight, reference)      AUC 0.779

LIGHTWEIGHT triage (age + BMI + menopause) - AUC: 0.867
Cutoff for 95% sensitivity: refer if P(osteoporosis) >= 0.029  (safely excludes 49% of women as low-risk)
   age 23, bmi 25, postmeno 0: P(osteoporosis) = 0.3%
   age 43, bmi 25, postmeno 0: P(osteoporosis) = 1.8%
   age 55, bmi 24, postmeno 1: P(osteoporosis) = 5.0%
   age 70, bmi 22, postmeno 1: P(osteoporosis) = 20.3%

HEAVYWEIGHT T-score model
 Metric  Value
    MAE   0.73
     R2   0.27
n_train 839.00
 n_test 280.00

                  Term  Coefficient      Direction
 

In [2]:
# --- 7. Lightweight triage threshold protocol ---
# Threshold selection uses validation data only. The held-out test set is used once,
# after the threshold is locked, for a retrospective research-screening audit.
from triage_audit import audit_threshold, select_threshold

X_train, X_temp, y_train, y_temp = train_test_split(
    Xl, yl, test_size=0.40, random_state=17, stratify=yl
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=17, stratify=y_temp
)

locked_triage = LogisticRegression(max_iter=1000).fit(X_train, y_train)
valid_probabilities = locked_triage.predict_proba(X_valid)[:, 1]
test_probabilities = locked_triage.predict_proba(X_test)[:, 1]

TARGET_SENSITIVITY = 0.95
CANDIDATE_THRESHOLDS = np.round(np.arange(0.01, 0.101, 0.01), 2)
validation_rows = []
for candidate in CANDIDATE_THRESHOLDS:
    candidate_audit = audit_threshold(y_valid, valid_probabilities, candidate, bootstrap_samples=0)
    validation_rows.append({
        "threshold": candidate,
        "sensitivity": candidate_audit["sensitivity"],
        "NPV": candidate_audit["npv"],
        "false negatives": candidate_audit["false_negatives"],
    })
validation_table = pd.DataFrame(validation_rows)
print("=== Validation-only threshold comparison (do not use test results here) ===")
print(validation_table.to_string(index=False, formatters={"sensitivity": "{:.1%}".format, "NPV": "{:.1%}".format}))

try:
    FINAL_TRIAGE_THRESHOLD = select_threshold(
        y_valid, valid_probabilities, CANDIDATE_THRESHOLDS, TARGET_SENSITIVITY
    )
except ValueError as error:
    raise RuntimeError("No candidate met the pre-specified validation safety target; do not deploy a routing threshold.") from error

final_audit = audit_threshold(
    y_test, test_probabilities, FINAL_TRIAGE_THRESHOLD, bootstrap_samples=2000, random_state=17
)
print(f"\nLocked triage threshold: {FINAL_TRIAGE_THRESHOLD:.0%} (selected on validation for ≥{TARGET_SENSITIVITY:.0%} sensitivity)")
print("=== Final held-out test audit at the locked threshold ===")
print(f"Sensitivity: {final_audit['sensitivity']:.1%} (95% CI {final_audit['sensitivity_ci95'][0]:.1%}–{final_audit['sensitivity_ci95'][1]:.1%})")
print(f"NPV: {final_audit['npv']:.1%} (95% CI {final_audit['npv_ci95'][0]:.1%}–{final_audit['npv_ci95'][1]:.1%})")
print(f"False negatives: {final_audit['false_negatives']} of {(y_test == 1).sum()} DXA-defined cases")
print(f"Brier score: {final_audit['brier_score']:.3f} (lower is better)")
print("Calibration bins (predicted risk vs observed prevalence):")
print(pd.DataFrame(final_audit["calibration_bins"]).query("count > 0").to_string(index=False))
print("Research screening only: this audit does not diagnose osteoporosis or establish a clinical referral rule.")

=== Validation-only threshold comparison (do not use test results here) ===
 threshold sensitivity    NPV  false negatives
      0.01      100.0% 100.0%                0
      0.02       95.7%  99.0%                1
      0.03       91.3%  98.6%                2
      0.04       87.0%  98.1%                3
      0.05       87.0%  98.2%                3
      0.06       87.0%  98.5%                3
      0.07       87.0%  98.5%                3
      0.08       73.9%  97.2%                6
      0.09       65.2%  96.5%                8
      0.10       65.2%  96.7%                8

Locked triage threshold: 2% (selected on validation for ≥95% sensitivity)
=== Final held-out test audit at the locked threshold ===
Sensitivity: 100.0% (95% CI 100.0%–100.0%)
NPV: 100.0% (95% CI 100.0%–100.0%)
False negatives: 0 of 24 DXA-defined cases
Brier score: 0.061 (lower is better)
Calibration bins (predicted risk vs observed prevalence):
 lower  upper  count  mean_predicted_risk  observed_preval